# S05E04 — Goingthere

**Zadanie:** Nawigować rakietą przez planszę 3×12 do bazy (kolumna 12).
Każdy krok: rozbrojenie radarów (frequencyScanner), hint nawigacyjny, wybór ruchu.

**Mechanika:**
- Plansza 3 rzędy (port/center/starboard), 12 kolumn
- Każda kolumna ma jeden kamień — hint mówi w którym rzędzie
- Komendy: `start`, `go`, `left`, `right`
- Radary: SHA1(detectionCode + 'disarm') → disarmHash

**Techniki:** iteracyjne próby, Claude Opus do parsowania hintów, SHA1 do rozbrojeń

In [ ]:
import anthropic
import requests
import hashlib
import time
import re
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

VERIFY_URL    = "https://hub.ag3nts.org/verify"
GETMESSAGE_URL = "https://hub.ag3nts.org/api/getmessage"
SCANNER_URL   = "https://hub.ag3nts.org/api/frequencyScanner"
API_KEY       = os.environ["AI_DEVS_API_KEY"]

client = anthropic.Anthropic()

print("Konfiguracja OK.")

In [ ]:
# ── FUNKCJE POMOCNICZE ───────────────────────────────────────────────────────

def sha1(text):
    return hashlib.sha1(text.encode()).hexdigest()


def post_verify(payload):
    r = requests.post(VERIFY_URL, json=payload,
                      headers={'Content-Type': 'application/json'}, timeout=15)
    return r.json()


def get_hint():
    """Pobierz hint nawigacyjny z retry na rate limit."""
    for attempt in range(5):
        try:
            r = requests.post(GETMESSAGE_URL, json={"apikey": API_KEY},
                              headers={'Content-Type': 'application/json'}, timeout=15)
            data = r.json()
            hint = ''
            if isinstance(data, dict):
                hint = data.get('hint', data.get('message', str(data)))
            else:
                hint = str(data)
            if 'często' in hint or 'zwolnij' in hint.lower() or 'slow down' in hint.lower():
                print(f"  RATE LIMIT: waiting 3s... ({hint[:50]})")
                time.sleep(3)
                continue
            return hint
        except Exception:
            time.sleep(2)
    return None


def scan_text():
    r = requests.get(f"{SCANNER_URL}?key={API_KEY}", timeout=10)
    return r.text


def disarm_radar(freq, hash_val):
    r = requests.post(SCANNER_URL, json={"apikey": API_KEY, "frequency": freq, "disarmHash": hash_val},
                      headers={'Content-Type': 'application/json'}, timeout=10)
    return r.json()


def is_clear_text(text):
    """Sprawdź czy skaner mówi 'clear' w dowolnej formie."""
    if len(text) > 300:
        return False
    if text.strip().startswith('<') or 'DOCTYPE' in text:
        return False
    return bool(re.search(r"it.{0,6}s\s*cl|clear", text, re.IGNORECASE))


print("Funkcje pomocnicze gotowe.")

In [ ]:
# ── SKANER RADARÓW ───────────────────────────────────────────────────────────

def parse_radar_response(text):
    """Claude ekstrahuje frequency + detectionCode z zagłuszonego JSON."""
    r = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=80,
        messages=[{'role': 'user', 'content':
            f'Extract frequency number and detectionCode string from this text. '
            f'Reply: FREQ=<number> CODE=<string>\nIf it says clear: reply CLEAR\n\n{text[:400]}'}]
    )
    resp = r.content[0].text.strip()
    if 'CLEAR' in resp.upper():
        return 'clear'
    fm = re.search(r'FREQ[=:]\s*([\d.]+)', resp, re.IGNORECASE)
    cm = re.search(r'CODE[=:]\s*(\S+)', resp, re.IGNORECASE)
    if fm and cm:
        try:
            return {'frequency': int(float(fm.group(1))), 'code': cm.group(1)}
        except ValueError:
            return None
    return None


def handle_scanner(player_row, player_col):
    """Poll skanera do czasu clear, rozbraja radary w razie potrzeby."""
    for _ in range(30):
        try:
            text = scan_text()
        except Exception:
            time.sleep(0.5)
            continue

        if not text or not text.strip():
            time.sleep(0.3)
            continue
        if text.strip().startswith('<') or 'DOCTYPE' in text.lower():
            time.sleep(0.3)
            continue

        if is_clear_text(text):
            print(f"  CLEAR: {text.strip()[:70]}")
            return

        result = parse_radar_response(text)
        if result == 'clear':
            print("  CLEAR (parsed)")
            return
        if result and 'frequency' in result:
            freq = result['frequency']
            code = result['code']
            h = sha1(code + 'disarm')
            print(f"  RADAR: freq={freq} code={code}")
            dr = disarm_radar(freq, h)
            print(f"  DISARMED: {str(dr)[:80]}")
            time.sleep(0.5)
            continue

        print(f"  SCANNER?: {text.strip()[:80]}")
        time.sleep(0.3)

    print("  WARNING: Scanner not clear after 30 attempts, proceeding")


print("handle_scanner() gotowe.")

In [ ]:
# ── LOGIKA NAWIGACJI ─────────────────────────────────────────────────────────

def get_stone_row(hint_text, player_row):
    """Claude Opus parsuje hint — w którym rzędzie jest kamień w kolejnej kolumnie.

    Reguły interpretacji:
    - "port" / "left side" = Row 1 (absolutny)
    - "starboard" / "right side" = Row 3 (absolutny)
    - "center" / "ahead" / "bow" = row gracza (względny)
    """
    prompt = f"""Rocket navigation hint analysis. Grid has 3 rows:
- Row 1 = PORT side (left, top)
- Row 2 = CENTER (middle)
- Row 3 = STARBOARD side (right, bottom)

Rocket is currently at Row {player_row}.

IMPORTANT RULES for interpreting hints:
- "port" / "port side" / "left side" = Row 1 (absolute)
- "starboard" / "starboard side" / "right side" = Row 3 (absolute)
- "center lane" / "middle" / "ahead" / "in front" / "bow" / "nose" / "straight" / "directly in front" = Row {player_row} (SAME AS ROCKET = relative/ahead)
- "both edges" / "both sides" = the rows that are NOT the player's row

Hint: "{hint_text}"

Where is the STONE/ROCK/OBSTACLE/BLOCKAGE in the next column?
Reply with ONLY one number: 1, 2, or 3"""

    r = client.messages.create(
        model='claude-opus-4-6',
        max_tokens=10,
        messages=[{'role': 'user', 'content': prompt}]
    )
    resp = r.content[0].text.strip()
    for ch in resp:
        if ch in '123':
            return int(ch)
    return player_row  # domyślnie: kamień przed nami


def choose_command(player_row, stone_row, base_row):
    """Wybierz ruch: go / left / right.
    
    left = row-1 (w kierunku port/row1)
    right = row+1 (w kierunku starboard/row3)
    
    Gdy kamień na row gracza: preferuj right (starboard) — bezpieczniejszy.
    """
    if stone_row == player_row:
        if player_row == 1:
            return 'right'
        elif player_row == 3:
            return 'left'
        else:  # player at row=2
            return 'right'  # starboard consistently safer
    else:
        if player_row == base_row:
            return 'go'
        elif player_row < base_row:
            new_row = player_row + 1
            return 'right' if new_row != stone_row else 'go'
        else:
            new_row = player_row - 1
            return 'left' if new_row != stone_row else 'go'


print("Logika nawigacji gotowa.")

In [ ]:
# ── JEDNA PRÓBA NAWIGACJI ────────────────────────────────────────────────────

def run_attempt(attempt_num):
    print(f"\n{'='*50}")
    print(f"ATTEMPT {attempt_num}")

    start = post_verify({"apikey": API_KEY, "task": "goingthere", "answer": {"command": "start"}})
    if 'player' not in start:
        print(f"Start failed: {start}")
        return None

    player_row = start['player']['row']
    player_col = start['player']['col']
    base_row   = start.get('base', {}).get('row', 1)
    print(f"Start: row={player_row} col={player_col} → Base row={base_row} col=12")

    for step in range(12):
        print(f"\n[row={player_row} col={player_col}]")

        # 1. Rozbrój radary
        handle_scanner(player_row, player_col)

        # 2. Pobierz hint
        time.sleep(0.5)
        hint_text = get_hint()
        if not hint_text:
            print("  WARNING: Could not get hint, using default strategy")
            hint_text = ""
        print(f"Hint: {hint_text[:120]}")

        # 3. Określ rząd kamienia
        stone_row = get_stone_row(hint_text, player_row) if hint_text else player_row
        print(f"Stone in next col: row={stone_row}")

        # 4. Wybierz komendę
        cmd = choose_command(player_row, stone_row, base_row)
        print(f"CMD: {cmd}  [player={player_row}, stone={stone_row}, base={base_row}]")

        # 5. Wykonaj ruch
        result = post_verify({"apikey": API_KEY, "task": "goingthere", "answer": {"command": cmd}})
        result_str = str(result)
        print(f"Result: {result_str[:150]}")

        # Sprawdź flagę
        flg = re.search(r'\{FLG:[^}]+\}', result_str)
        if flg:
            print(f"\n*** FLAG: {flg.group(0)} ***")
            return flg.group(0)

        if isinstance(result, dict) and result.get('crashed'):
            print(f"CRASHED! Reason: {result.get('crashReason', '?')}")
            return None

        if isinstance(result, dict) and 'player' in result:
            player_row = result['player']['row']
            player_col = result['player']['col']

            if player_col >= 12:
                print(f"Reached col=12! Result: {result_str}")
                final = post_verify({"apikey": API_KEY, "task": "goingthere", "answer": {"command": "go"}})
                print(f"Final: {final}")
                flg2 = re.search(r'\{FLG:[^}]+\}', str(final))
                if flg2:
                    return flg2.group(0)
                return result_str
        else:
            print(f"Unexpected: {result_str}")
            return None

    print(f"Completed moves. Last pos: row={player_row} col={player_col}")
    return None


print("run_attempt() gotowe.")

In [ ]:
# ── PĘTLA GŁÓWNA — wielokrotne próby do flagi ────────────────────────────────

flag = None
for attempt in range(1, 100):
    result = run_attempt(attempt)
    if result:
        if '{FLG:' in str(result):
            flag = result
            print(f"\n=== FINAL FLAG: {flag} ===")
            break
        print(f"Non-flag result: {result}")
    time.sleep(1)

print(f"\n=== FINAL FLAG: {flag} ===")